In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

# ==========================================
# 1. MOTOR MATEMÁTICO GENÉRICO
# ==========================================
def avaliar_grid(params, var1_name, var1_array, var2_name, var2_array):
    V1, V2 = np.meshgrid(var1_array, var2_array, indexing='ij')

    p = params.copy()
    p[var1_name] = V1
    p[var2_name] = V2

    g1 = p['P'] * (p['R'] - p['LR'] - p['Cs'] - p['G']) + (1 - p['P']) * (-p['Cs'])
    g2 = (1 - p['P']) * (p['I'] - p['Ca']) + p['P'] * (p['I'] - p['L'] + p['G'] - p['Ca'])

    n_IA = (p['R'] >= 0) & ((p['I'] - p['L']) >= g2)
    n_IF = (g1 >= 0) & (g2 >= (p['I'] - p['L']))
    n_WA = (0 >= p['R']) & (p['I'] >= (p['I'] - p['alpha'] * p['Ca']))
    n_WF = (0 >= g1) & ((p['I'] - p['alpha'] * p['Ca']) >= p['I'])

    IA = n_IA
    IF = n_IF & ~IA
    WA = n_WA & ~IA & ~IF
    WF = n_WF & ~IA & ~IF & ~WA

    return IA, IF, WA, WF

def aplicar_variacao(val_base, var_array, tipo_var):
    if tipo_var == 'Percentual (%)':
        return val_base * (1 + var_array / 100.0)
    return var_array

def achar_cruzamento(x, y1, y2):
    diff = np.array(y1) - np.array(y2)
    cruzamentos = []
    for i in range(len(diff) - 1):
        if diff[i] * diff[i+1] < 0:
            x_int = x[i] - diff[i] * (x[i+1] - x[i]) / (diff[i+1] - diff[i])
            y_int = y1[i] + (y1[i+1] - y1[i]) * (x_int - x[i]) / (x[i+1] - x[i])
            cruzamentos.append((x_int, y_int))
    return cruzamentos

cores = {'IA': '#4CAF50', 'IF': '#F44336', 'WA': '#9E9E9E', 'WF': '#607D8B', 'NoEq': '#FFC107'}
labels_eq = {'IA': 'Implement / Accept', 'IF': 'Implement / Fight', 'WA': 'Withdraw / Accept', 'WF': 'Withdraw / Fight', 'NoEq': 'Absence of Pure Equilibrium'}

nomes_bonitos = {
    'R': "Consortium Expected Return 'R'",
    'LR': "Delay Cost 'LR' (in Millions)",
    'Cs': "Judicial Cost 'Cs' (in Millions)",
    'I': "Stakeholders Base Revenue 'I'",
    'L': "Estimated Local Economic Impact 'L'",
    'alpha': "Mobilization Cost Coefficient 'alpha'",
    'Ca': "Judicial Cost 'Ca' (in Millions)",
    'G': "Financial Compensation Offered 'G'",
    'P': "Probability of Judicial Success 'P'"
}

# ==========================================
# 2. INTERFACE DE USUÁRIO (DASHBOARD)
# ==========================================
style = {'description_width': '180px'}
layout = {'width': '450px'}
variaveis = ['R', 'LR', 'Cs', 'I', 'L', 'alpha', 'Ca', 'G', 'P']

lbl_cat1 = widgets.HTML("<h3 style='margin-bottom:0;'>1. Seleção da Análise</h3><hr style='margin:0 0 10px 0;'>")
w_tipo_grafico = widgets.Dropdown(options=['Evolução de Áreas (Grid 2D vs Var X)', 'Mapa de Calor 2D (Var Y vs Var X)', 'Payoff Esperado (Surf 1D)'], value='Evolução de Áreas (Grid 2D vs Var X)', description='Tipo de Gráfico:', style=style, layout=layout)
w_impacto = widgets.Dropdown(options=[('Baixo (L=4.4)', 4.4), ('Médio (L=8.8)', 8.8), ('Alto (L=19.8)', 19.8)], value=4.4, description='Cenário Impacto (L):', style=style, layout=layout)
box_cat1 = widgets.VBox([lbl_cat1, w_tipo_grafico, w_impacto], layout={'margin': '0 0 20px 0'})

lbl_cat2 = widgets.HTML("<h3 style='margin-bottom:0;'>2. Eixo Principal (Variável X)</h3><hr style='margin:0 0 10px 0;'>")
w_var_x = widgets.Dropdown(options=variaveis, value='Ca', description='Qual Variável?', style=style, layout=layout)
w_tipo_var_x = widgets.Dropdown(options=['Absoluto', 'Percentual (%)'], value='Absoluto', description='Tipo de Variação:', style=style, layout=layout)
w_min_x = widgets.FloatText(value=0.0, description='Valor Mínimo:', style=style, layout=layout)
w_max_x = widgets.FloatText(value=22.0, description='Valor Máximo:', style=style, layout=layout)
w_steps_x = widgets.IntText(value=21, description='Qtd. Pontos da Linha:', style=style, layout=layout)
box_cat2 = widgets.VBox([lbl_cat2, w_var_x, w_tipo_var_x, w_min_x, w_max_x, w_steps_x], layout={'margin': '0 0 20px 0', 'border': '1px solid #ddd', 'padding': '10px'})

lbl_cat3 = widgets.HTML("<h3 style='margin-bottom:0;'>3. Matriz de Fundo (Grids Y e Z)</h3><hr style='margin:0 0 10px 0;'>")
w_var_y = widgets.Dropdown(options=variaveis, value='P', description='Eixo Y / Grid 1:', style=style, layout=layout)
w_min_y = widgets.FloatText(value=0.0, description='Mínimo Y/Grid1:', style=style, layout=layout)
w_max_y = widgets.FloatText(value=1.0, description='Máximo Y/Grid1:', style=style, layout=layout)
w_var_z = widgets.Dropdown(options=variaveis, value='G', description='Grid 2 (Só p/ Áreas):', style=style, layout=layout)
w_min_z = widgets.FloatText(value=0.0, description='Mínimo Grid2:', style=style, layout=layout)
w_max_z = widgets.FloatText(value=6.6, description='Máximo Grid2:', style=style, layout=layout)
w_res = widgets.IntText(value=100, description='Resolução da Matriz:', style=style, layout=layout)
box_cat3 = widgets.VBox([lbl_cat3, w_var_y, w_min_y, w_max_y, widgets.HTML("<br>"), w_var_z, w_min_z, w_max_z, widgets.HTML("<br>"), w_res], layout={'margin': '0 0 20px 0', 'border': '1px solid #ddd', 'padding': '10px'})

aba_config = widgets.VBox([box_cat1, box_cat2, box_cat3])

w_R = widgets.FloatText(value=100.0, description='Retorno (R):', style=style, layout=layout)
w_LR = widgets.FloatText(value=20.0, description='Custo Atraso (LR):', style=style, layout=layout)
w_Cs = widgets.FloatText(value=2.0, description='Custo Jud. Cons. (Cs):', style=style, layout=layout)
w_I = widgets.FloatText(value=22.0, description='Receita Surf (I):', style=style, layout=layout)
w_alpha = widgets.FloatText(value=0.1, description='Mobilização (alpha):', style=style, layout=layout)
w_Ca = widgets.FloatText(value=4.0, description='Custo Jud. Surf (Ca):', style=style, layout=layout)
w_G = widgets.FloatText(value=0.0, description='Compensação (G):', style=style, layout=layout)
w_P = widgets.FloatText(value=0.5, description='Prob. Vitória (P):', style=style, layout=layout)

aba_params = widgets.VBox([widgets.HTML("<h3>Parâmetros Estáticos do Modelo</h3><hr>"), w_R, w_LR, w_Cs, w_I, w_Ca, w_alpha, w_G, w_P], layout={'padding': '10px'})

abas = widgets.Tab(children=[aba_config, aba_params])
abas.set_title(0, 'Controle do Gráfico')
abas.set_title(1, 'Parâmetros Fixos')

# ==========================================
# 3. ATUALIZAÇÃO AUTOMÁTICA
# ==========================================
def obter_limites_padrao(var_name, L_val, I_val, tipo_var):
    if tipo_var == 'Percentual (%)': return -50.0, 50.0
    if var_name in ['P', 'alpha']: return 0.0, 1.0
    if var_name == 'G': return 0.0, 1.5 * L_val
    if var_name == 'Ca': return 0.0, I_val
    if var_name in ['LR', 'Cs']: return 0.0, 100.0
    if var_name == 'L': return 0.0, 19.8
    if var_name in ['R', 'I']: return 0.0, 200.0
    return 0.0, 100.0

def atualizar_limites(*args):
    L_val, I_val = w_impacto.value, w_I.value
    w_min_x.value, w_max_x.value = obter_limites_padrao(w_var_x.value, L_val, I_val, w_tipo_var_x.value)
    w_min_y.value, w_max_y.value = obter_limites_padrao(w_var_y.value, L_val, I_val, 'Absoluto')
    w_min_z.value, w_max_z.value = obter_limites_padrao(w_var_z.value, L_val, I_val, 'Absoluto')

for w in [w_var_x, w_var_y, w_var_z, w_tipo_var_x, w_impacto, w_I]: w.observe(atualizar_limites, 'value')
atualizar_limites()

# ==========================================
# 4. BOTÕES E LÓGICA DE PLOTAGEM
# ==========================================
btn_plot = widgets.Button(description='Gerar Gráfico', button_style='primary', icon='play')
btn_export = widgets.Button(description='Exportar PNG', button_style='success', icon='download')
w_insight = widgets.HTML(value="", layout={'margin': '15px 0 0 0', 'padding': '10px', 'border-left': '4px solid #4CAF50'})
output = widgets.Output()
figura_atual = None

def plotar_grafico(b):
    global figura_atual
    with output:
        clear_output(wait=True)
        w_insight.value = ""

        params_base = {'R': w_R.value, 'LR': w_LR.value, 'Cs': w_Cs.value, 'I': w_I.value, 'L': w_impacto.value, 'alpha': w_alpha.value, 'Ca': w_Ca.value, 'G': w_G.value, 'P': w_P.value}

        figura_atual, ax = plt.subplots(figsize=(8, 6))
        x_array_linhas = np.linspace(w_min_x.value, w_max_x.value, w_steps_x.value)
        x_array_hm = np.linspace(w_min_x.value, w_max_x.value, w_res.value)

        nome_x = nomes_bonitos.get(w_var_x.value, w_var_x.value)
        sufixo = '(%)' if w_tipo_var_x.value == 'Percentual (%)' else ''
        prefixo = 'Absolute Value of ' if w_tipo_var_x.value == 'Absoluto' else 'Variation of '

        if w_tipo_grafico.value == 'Evolução de Áreas (Grid 2D vs Var X)':
            grid1_array = np.linspace(w_min_y.value, w_max_y.value, w_res.value)
            grid2_array = np.linspace(w_min_z.value, w_max_z.value, w_res.value)

            areas = {'IA': [], 'IF': [], 'WA': [], 'WF': [], 'NoEq': []}
            x_vals = aplicar_variacao(params_base[w_var_x.value], x_array_linhas, w_tipo_var_x.value)

            for x_val in x_vals:
                p_iter = params_base.copy()
                p_iter[w_var_x.value] = x_val
                IA, IF, WA, WF = avaliar_grid(p_iter, w_var_y.value, grid1_array, w_var_z.value, grid2_array)
                NoEq = ~IA & ~IF & ~WA & ~WF
                total = w_res.value ** 2
                areas['IA'].append(np.sum(IA) / total * 100)
                areas['IF'].append(np.sum(IF) / total * 100)
                areas['WA'].append(np.sum(WA) / total * 100)
                areas['WF'].append(np.sum(WF) / total * 100)
                areas['NoEq'].append(np.sum(NoEq) / total * 100)

            for key in ['IA', 'IF', 'WA', 'WF', 'NoEq']:
                if max(areas[key]) > 0:
                    ax.plot(x_array_linhas, areas[key], color=cores[key], marker='o', markersize=5, linewidth=2.5, label=labels_eq[key])

            # Insight limpo, usando tags semutais HTML em vez de cores inline rigorosas
            cruzamentos = achar_cruzamento(x_array_linhas, areas['IA'], areas['IF'])
            if cruzamentos:
                texto_cruzamento = "".join([f"<li>Quando <b>{w_var_x.value} = {c_x:.2f}</b>, as áreas se igualam em <b>{c_y:.1f}%</b>.</li>" for c_x, c_y in cruzamentos])
                w_insight.value = f"<div style='font-size:14px;'><b>Ponto de Transição Estratégica:</b> Onde o acordo (Verde) e o litígio (Vermelho) possuem o mesmo peso/probabilidade no cenário de incertezas.<ul>{texto_cruzamento}</ul></div>"

            ax.set_xlabel(f"{prefixo}{nome_x} {sufixo}")
            ax.set_ylabel(f"Area in the {w_var_y.value} vs {w_var_z.value} scenario grid (%)")
            ax.set_ylim(-5, 105)
            ax.set_xticks(np.linspace(w_min_x.value, w_max_x.value, 11))
            ax.legend(title="Nash Equilibrium", loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=True, edgecolor='black')

        elif w_tipo_grafico.value == 'Mapa de Calor 2D (Var Y vs Var X)':
            y_array = np.linspace(w_min_y.value, w_max_y.value, w_res.value)
            IA, IF, WA, WF = avaliar_grid(params_base, w_var_y.value, y_array, w_var_x.value, x_array_hm)
            NoEq = ~IA & ~IF & ~WA & ~WF

            Z = np.zeros((w_res.value, w_res.value))
            Z[IA], Z[IF], Z[WA], Z[WF], Z[NoEq] = 0, 1, 2, 3, 4
            cmap = mcolors.ListedColormap([cores['IA'], cores['IF'], cores['WA'], cores['WF'], cores['NoEq']])
            norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

            X_mesh, Y_mesh = np.meshgrid(x_array_hm, y_array)
            ax.pcolormesh(X_mesh, Y_mesh, Z, cmap=cmap, norm=norm, shading='auto')

            w_insight.value = "<div style='font-size:14px;'><b>Leitura do Mapa:</b> Gráficos de calor não possuem pontos únicos de cruzamento de linhas. A fronteira entre as cores indica exatamente a zona de transição de decisão dos atores.</div>"

            ax.set_xlabel(nomes_bonitos.get(w_var_x.value, w_var_x.value))
            ax.set_ylabel(nomes_bonitos.get(w_var_y.value, w_var_y.value))
            chaves = ['IA', 'IF', 'WA', 'WF', 'NoEq']
            legend_elements = [Patch(facecolor=cores[chaves[int(v)]], edgecolor='black', label=labels_eq[chaves[int(v)]]) for v in np.unique(Z)]
            ax.legend(handles=legend_elements, title="Nash Equilibrium", loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=True, edgecolor='black')

        else:
            x_vals = aplicar_variacao(params_base[w_var_x.value], x_array_linhas, w_tipo_var_x.value)
            payoff_acc, payoff_fig = [], []
            for x_val in x_vals:
                p_iter = params_base.copy()
                p_iter[w_var_x.value] = x_val
                acc = p_iter['I'] - p_iter['L']
                fig = (1 - p_iter['P']) * (p_iter['I'] - p_iter['Ca']) + p_iter['P'] * (p_iter['I'] - p_iter['L'] + p_iter['G'] - p_iter['Ca'])
                payoff_acc.append(acc)
                payoff_fig.append(fig)

            ax.plot(x_array_linhas, payoff_acc, 'k--', linewidth=2, label='Agreement')
            ax.plot(x_array_linhas, payoff_fig, 'gray', linewidth=2, label='Fight')

            cruzamentos = achar_cruzamento(x_array_linhas, payoff_acc, payoff_fig)
            if cruzamentos:
                texto_cruzamento = "".join([f"<li><b>{w_var_x.value} = {c_x:.2f}</b> (Resultando em um payoff esperado de <b>{c_y:.1f}</b>).</li>" for c_x, c_y in cruzamentos])
                w_insight.value = f"<div style='font-size:14px;'><b>Ponto de Indiferença (Price of Peace):</b> Onde o valor esperado entre Aceitar e Lutar é idêntico:<ul>{texto_cruzamento}</ul></div>"

            ax.set_xlabel(f"{prefixo}{nome_x} {sufixo}")
            ax.set_ylabel('Expected Payoff of the Surf Community')
            ax.set_xticks(np.linspace(w_min_x.value, w_max_x.value, 11))
            ax.legend(title="Rational Decision", loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=True, edgecolor='black')

        ax.grid(True, linestyle='--', alpha=0.6)
        figura_atual.tight_layout()
        plt.show()

def exportar_grafico(b):
    global figura_atual
    with output:
        if figura_atual is not None:
            filename = "grafico_pesquisa_colab.png"
            figura_atual.savefig(filename, dpi=300, bbox_inches='tight')
            files.download(filename)
        else:
            print("Gere um gráfico primeiro!")

btn_plot.on_click(plotar_grafico)
btn_export.on_click(exportar_grafico)

display(abas, widgets.HTML("<br>"), widgets.HBox([btn_plot, btn_export]), w_insight, output)

HTML(value='<br>')

HTML(value='', layout=Layout(margin='15px 0 0 0', padding='10px'))

Output()